In [1]:
# Pandas pour manipuler les tableaux de données
import pandas as pd
pd.set_option('display.max_columns', 500)

# Numpy pour les listes de données numériques et les fonctions classiques mathématiques
import numpy as np

# scipy (librairie scientifique) pour les fonctions statistiques et autres utilisaires
import scipy

# scikit learn pour les outils de machine learning
import sklearn
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler

# librairies pour la visualisation de données
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# et quelques options visuelles
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline
sns.set(style="whitegrid", color_codes=True)
sns.set(rc={'figure.figsize':(15,8)})

In [2]:
# cette fonction évalue la corrélation entre variables qualitatives en 
# - élaboration du tableau de contingence des valeurs
# - calcul du chi2 de cet tableau 
# - calcul du coefficient de cramer qui est une normalisation du coefficient chi2
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x,y)
    chi2 = scipy.stats.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2/n
    r,k = confusion_matrix.shape
    phi2corr = max(0, phi2-((k-1)*(r-1))/(n-1))
    rcorr = r-((r-1)**2)/(n-1)
    kcorr = k-((k-1)**2)/(n-1)
    return np.sqrt(phi2corr/min((kcorr-1),(rcorr-1)))

In [3]:
# élaboration de deux listes de variables aléatoires
x = np.random.randint(0,20, size=1000)
y = np.random.randint(0,20, size=1000)

In [4]:
# test pour deux listes décorrélées
cramers_v(x,y)

np.float64(0.063330685836234)

In [5]:
# test pour deux listes totalement corrélées
cramers_v(x,x)

np.float64(1.0)

In [6]:
def eta_squared(x,y):
    moyenne_y = y.mean()
    classes = []
    for classe in x.unique():
        yi_classe = y[x==classe]
        classes.append({'ni': len(yi_classe),
                        'moyenne_classe': yi_classe.mean()})
    SCT = sum([(yj-moyenne_y)**2 for yj in y])
    SCE = sum([c['ni']*(c['moyenne_classe']-moyenne_y)**2 for c in classes])
    return SCE/SCT

In [7]:
# les données ont été téléchargées sur kaggle et enregistrées en local 
# dans un répertoire similaire que celui en ligne sur kaggle, ainsi le script
# pourra tourner aussi bien en ligne qu'en local
train = pd.read_csv('../data/train.csv')
test =  pd.read_csv('../data/test.csv')
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [8]:
# affichage des 5 premières lignes du jeu de test
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [9]:
test['Survived']=np.nan
data=pd.concat([train,test],keys=['train','test'], join='inner')
data.index=data.index.droplevel(level=1)

In [10]:
len(data)

1309

In [11]:
len(data.loc['test'])

418